# Cross-Position Interaction Analysis

How information flows between token positions: pairwise influence matrices, directional transfer tracing, position importance ranking, interaction clustering, and critical information path identification.

## Why This Matters

Cross-component position interaction measures how information transfers between different parts of the model. Understanding cross-component interactions reveals the collaborative computation that produces emergent model capabilities.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.cross_position_interaction import (
    pairwise_position_influence,
    directional_information_transfer,
    position_importance_ranking,
    interaction_clustering,
    critical_information_path,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
# Pairwise position influence
result = pairwise_position_influence(model, tokens, layer=0)
print(f'Influence matrix shape: {result["influence_matrix"].shape}')
print(f'Most influential source: pos {result["most_influential_source"]}')
print(f'Most influenced target: pos {result["most_influenced_target"]}')
print('Strongest pairs (dst, src, weight):')
for dst, src, w in result['strongest_pairs'][:5]:
    print(f'  {dst} <- {src}: {w:.4f}')

In [ ]:
# Directional transfer from first to last position
result = directional_information_transfer(model, tokens, source_pos=0, target_pos=-1)
print(f'Peak transfer layer: {result["peak_transfer_layer"]}')
print(f'Cumulative transfer: {result["cumulative_transfer"]:.4f}')
for layer in result['per_layer']:
    print(f'  Layer {layer["layer"]}: mean={layer["mean_transfer"]:.4f}, '
          f'max={layer["max_transfer"]:.4f} (head {layer["max_transfer_head"]})')

In [ ]:
# Position importance ranking for the last position
result = position_importance_ranking(model, tokens)
print('Ranked positions:')
for pos, score in result['ranked_positions']:
    print(f'  Pos {pos}: importance={score:.4f}')

In [ ]:
# Cluster positions by interaction patterns
result = interaction_clustering(model, tokens, layer=0, n_clusters=3)
print(f'Cluster sizes: {result["cluster_sizes"]}')
print(f'Assignments: {np.array(result["cluster_assignments"])}')
print(f'Within-cluster similarity: {[f"{s:.3f}" for s in result["within_cluster_similarity"]]}')

In [ ]:
# Critical information paths to the last position
result = critical_information_path(model, tokens, threshold=0.1)
print(f'Active paths: {result["n_active_paths"]}')
print('Top paths:')
for p in result['paths'][:5]:
    print(f'  {" -> ".join(str(x) for x in p["path"])}: strength={p["strength"]:.4f}')
print(f'Critical positions: {result["critical_positions"]}')